# Corpus inventory and lightweight Markdown checks

Inventories the supplied PDFs and existing Markdown without running OCR, changing source files, creating embeddings, building a retrieval index, or calling a model. Clinical authority and transcription accuracy remain unassessed.

## Save configuration

Corpus manifest, file inventory CSV, issue CSV and validation results.

Set the output directory below before running. Each export creates a new run subdirectory beneath it and returns its actual path. Saving happens when the export/run cell executes; the script receives this directory explicitly. Existing files are not overwritten. Changing output paths does not change downstream input discovery automatically; pass explicit bundle/index paths when using custom locations.

In [1]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'pyproject.toml').exists())
OUTPUT_ROOT = ROOT / 'artifacts' / '01_corpus_inventory'
print('Save directory:', OUTPUT_ROOT.resolve())

Save directory: C:\Users\PK\Desktop\projects\mobile_rag\artifacts\01_corpus_inventory


In [2]:
from pathlib import Path
import json, sys

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'pyproject.toml').exists()), None)
if not (ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate the project root')
sys.path.insert(0, str(ROOT / 'src'))
from mobile_rag.corpus import build_inventory, export_inventory, validate_inventory
print({'project_root': str(ROOT), 'python': sys.version.split()[0]})

{'project_root': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag', 'python': '3.11.15'}


In [3]:
inventory = build_inventory(ROOT)
checks = validate_inventory(inventory)
counts = {
    'all_tracked_data_files': len(inventory['files']),
    'pdf_paths': sum(f['extension'] == '.pdf' and f['expected_location'] for f in inventory['files']),
    'unique_pdf_contents': len(inventory['pdf_contents']),
    'markdown_paths': len(inventory['extractions']),
    'matched_markdown': sum(p['status'] == 'matched' for p in inventory['pairings']),
    'issues': len(inventory['issues']),
}
print(counts)
print(checks)

{'all_tracked_data_files': 31, 'pdf_paths': 14, 'unique_pdf_contents': 14, 'markdown_paths': 14, 'matched_markdown': 14, 'issues': 3}
{'passed': True, 'checks': {'unique_file_ids': True, 'hashes_valid': True, 'sizes_nonnegative': True, 'pdf_file_references_valid': True, 'extraction_file_references_valid': True, 'pairing_file_references_valid': True}, 'clinical_review_status': 'not_assessed'}


In [4]:
pairing_view = []
file_by_id = {f['file_id']: f for f in inventory['files']}
for pairing in inventory['pairings']:
    pairing_view.append({
        'markdown': file_by_id[pairing['markdown_file_id']]['relative_path'],
        'status': pairing['status'],
        'rule': pairing['matching_rule'],
        'pdf_content_id': pairing['pdf_content_id'],
    })
pairing_view

[{'markdown': 'data/md_docs/Bubble-CPAP-guidelines-2017.md',
  'status': 'matched',
  'rule': 'explicit_source',
  'pdf_content_id': 'pdf_97a92d4bd3fc47e0dd87b739978826a8418d7c01b52175b5aeb4d7a70dd107a1'},
 {'markdown': 'data/md_docs/Child-Health-for-Nurses-and-HEOs-in-Papua-New-Guinea-3th-Edition-March-2022.md',
  'status': 'matched',
  'rule': 'explicit_source',
  'pdf_content_id': 'pdf_a7f8afddceae65b19ba807e4662dece6b18d2ff366110d1cf26847f95fd03cf7'},
 {'markdown': 'data/md_docs/Feeding Chart.md',
  'status': 'matched',
  'rule': 'explicit_source',
  'pdf_content_id': 'pdf_61d8d0567786420e6ffb7082e4fc3d449fab37ccf6782538f7edab96bd75e849'},
 {'markdown': 'data/md_docs/HIV Treatment Guidelines 2019.md',
  'status': 'matched',
  'rule': 'explicit_source',
  'pdf_content_id': 'pdf_f9c67a1ab5e7b8e57e01fd036f55188bdbc47573764f31c20c0a98c4fd0eb514'},
 {'markdown': 'data/md_docs/Malnutrition Treatment Chart.md',
  'status': 'matched',
  'rule': 'explicit_source',
  'pdf_content_id': 'pdf_d

In [5]:
markdown_view = [{
    'path': file_by_id[e['file_id']]['relative_path'],
    'characters': e['character_count'],
    'headings': e['heading_counts'],
    'pages': len(e['page_markers']),
    'tables': e['table_count'],
    'flags': e['structural_flags'],
} for e in inventory['extractions']]
markdown_view

[{'path': 'data/md_docs/Bubble-CPAP-guidelines-2017.md',
  'characters': 15636,
  'headings': {'1': 2, '2': 10, '3': 11},
  'pages': 10,
  'tables': 3,
  'flags': []},
 {'path': 'data/md_docs/Child-Health-for-Nurses-and-HEOs-in-Papua-New-Guinea-3th-Edition-March-2022.md',
  'characters': 799807,
  'headings': {'1': 1, '2': 413},
  'pages': 413,
  'tables': 4,
  'flags': []},
 {'path': 'data/md_docs/Feeding Chart.md',
  'characters': 6392,
  'headings': {'1': 1, '2': 3, '3': 3},
  'pages': 3,
  'tables': 4,
  'flags': []},
 {'path': 'data/md_docs/HIV Treatment Guidelines 2019.md',
  'characters': 219314,
  'headings': {'1': 1, '2': 132},
  'pages': 132,
  'tables': 1,
  'flags': []},
 {'path': 'data/md_docs/Malnutrition Treatment Chart.md',
  'characters': 1370,
  'headings': {'1': 1, '2': 2, '3': 1},
  'pages': 1,
  'tables': 1,
  'flags': []},
 {'path': 'data/md_docs/NDoH Updated For Children.md',
  'characters': 4192,
  'headings': {'1': 1, '2': 1, '3': 4},
  'pages': 1,
  'tables': 

In [6]:
issue_view = [{k: row[k] for k in ('code', 'severity', 'details')} for row in inventory['issues']]
issue_view

[{'code': 'markdown_structure',
  'severity': 'warning',
  'details': 'duplicate_page_markers:3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66'},
 {'code': 'markdown_structure',
  'severity': 'warning',
  'details': 'out_of_order_page_markers'},
 {'code': 'unexpected_location',
  'severity': 'warning',
  'details': 'data/questions/FrontlineAI_100_Clinically_Grounded_QA_Benchmark.pdf'}]

In [7]:
output_dir = export_inventory(inventory, ROOT, output_root=OUTPUT_ROOT)
saved_checks = json.loads((output_dir / 'check_results.json').read_text(encoding='utf-8'))
assert saved_checks['passed']
print({'output_dir': output_dir.resolve().as_posix(), 'input_fingerprint': inventory['run']['input_fingerprint'], 'checks_passed': True})

{'output_dir': 'C:/Users/PK/Desktop/projects/mobile_rag/artifacts/01_corpus_inventory/2026-09-13T11-33-18+00-00_d6d0852754', 'input_fingerprint': 'd6d0852754a2307b53ca4a51031f61f138abcfb48ddfcdce7b24b7401ff65375', 'checks_passed': True}


## Checkpoint

Technical inventory completion is separate from corpus clinical approval. Review the saved issue report before Markdown chunking. Detailed OCR/content auditing remains skipped.